In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from nrb_canonicalize import (
    find_candidate_duplicate_columns, coalesce_columns,
    canonicalize_line_item_regex, resolve_ambiguous_fragments,
    COMMERCIAL_BANKS, normalize_bfi_name, _PANEL_ENTITY_MAP,
)
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# warnings.filterwarnings("ignore", category=DtypeWarning, module="pandas")

# 1. Robust Path Detection for both .py and .ipynb
try:
    # Works in standard .py scripts
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for Jupyter Notebooks
    base_dir = Path.cwd()

# Locate data directory relative to project structure
if base_dir.name == "models":
    data_dir = base_dir.parent / "data" / "processed" / "monthly_statistical_bfis"
else:
    data_dir = base_dir / "data" / "processed" / "monthly_statistical_bfis"

csv_files = list(data_dir.glob("*.csv"))
print(f"Found {[str(i).split('/')[-1] for i in csv_files]} CSV files in: {data_dir}")
# 2. Iterate and Inspect CSVs
for file in csv_files:
    df = pd.read_csv(file)
    print(f"\nInspecting file: {file.name} {df.shape}")
    print("--" * 60)


In [ ]:
nrb_profit_and_loss = pd.read_csv(data_dir / "nrb_profit_and_loss.csv")
set(nrb_profit_and_loss.line_item.values.tolist()[:10])

In [ ]:
import numpy as np
import pandas as pd


nrb_balance_sheet=pd.read_csv(data_dir / "nrb_balance_sheet.csv")
nrb_balance_sheet.head(10)
commbanklist = [
    'ADBNL', 'ADBL', 'NABIL', 'NIBL', 'NIBL ACE', 'NMB', 'PRABHU',
    'SANIMA', 'SBI', 'SCB', 'HBL', 'CITIZEN', 'CBBL', 'CBBL HFL',
    'EBL', 'EBL NIBL', 'GBIME', 'HIMALAYAN', 'KUMARI', 'MBL', 'MBNL'
]

# 2. Filter raw balance sheet data for commercial banks only
nrb_balance_sheet= nrb_balance_sheet[nrb_balance_sheet['bfi_name'].isin(commbanklist)].copy()
nrb_balance_sheet.line_item.unique().tolist()[:10]

In [ ]:
nrb_balance_sheet = pd.read_csv(data_dir / "nrb_balance_sheet.csv")

# 1. Filter raw balance sheet data for commercial banks only
mask = nrb_balance_sheet['bfi_name'].isin(COMMERCIAL_BANKS.keys())
nrb_balance_sheet = nrb_balance_sheet[mask].copy()

nrb_balance_sheet['bfi_name'] = nrb_balance_sheet['bfi_name'].map(normalize_bfi_name)
nrb_balance_sheet['line_item'] = nrb_balance_sheet['line_item'].apply(canonicalize_line_item_regex)
nrb_balance_sheet = resolve_ambiguous_fragments(nrb_balance_sheet)
nrb_balance_sheet['value_mn_npr'] = pd.to_numeric(nrb_balance_sheet['value_mn_npr'], errors='coerce')

# 2. drop junk rows before pivoting, so they can't become spurious columns
drop_labels = {'DROP_TOTAL_ROW', 'DROP_FRAGMENT', 'UNKNOWN'}
nrb_balance_sheet = nrb_balance_sheet[
    ~nrb_balance_sheet['line_item'].isin(drop_labels)
    & ~nrb_balance_sheet['line_item'].str.startswith('UNKNOWN SECTION >', na=False)
]

pivot = nrb_balance_sheet.pivot_table(index=['year_bs', 'month_bs', 'bfi_name'],
                                        columns='line_item', values='value_mn_npr',
                                        aggfunc='sum').reset_index()

from nrb_canonicalize import build_paper1_features
paper1_features = build_paper1_features(pivot)

legacy_cols = [
    '2.f Borrowings - Govt Non-Financial Institutions',
    '2.g Borrowings - Other Non-Financial Institutions',
    '1.c Proposed Bonus Share',
    '1.a.1 Cash Balance - Cash in Transit',
    '6 LOANS AGAINST COLLECTED BILLS',
    '6 RECONCILIATION ACCOUNT',
    '6.a Loans Against Collected Bills - Domestic/Foreign',
    '8 CALLS IN ADVANCE',
    '2.d Other Financial Institutions',
    '2.e Bonds and Securities / NRB Bonds',
    '9 EXPENSES NOT WRITTEN OFF',
    '4 LOANS & ADVANCES > FINANCIAL',
    '4 LOANS & ADVANCES > GOVERNMENT',
    '3.e Deposits - Non Residents',
]
pivot = pivot.drop(columns=[c for c in legacy_cols if c in pivot.columns])

pivot.head(2)

In [ ]:
nrb_balance_sheet[nrb_balance_sheet['line_item'] == 'b. Interbank Borrowing'].head(1)

nrb_balance_sheet.line_item.unique().tolist()[:2]

In [ ]:
nrb_balance_sheet[nrb_balance_sheet['line_item'] == 'a. Interbank Lending'].head(1)

In [ ]:
nrb_balance_sheet.pivot_table(
    index=['source_file', 'year_bs', 'month_bs', 'bfi_name'],
    columns='line_item',
    values='value_mn_npr',
    aggfunc='sum'
).reset_index().head(2)

In [ ]:

# 3. Define the exact raw line items we need to extract from the dataset
target_raw_items = [
    '1 CAPITAL FUND',
    '1 LIQUID FUNDS',
    '3 DEPOSITS',
    '4 LOANS & ADVANCES',
    '4.1 LOANS & ADVANCES',
    '2 BORROWINGS',
    'a. Interbank Lending',
    'b. Interbank Borrowing',
    'a. NRB'  # Under borrowings
]

# 4. Extract only the rows matching our target raw items
nrb_balance_sheet = nrb_balance_sheet[nrb_balance_sheet['line_item'].str.strip().isin(target_raw_items)].sort_values(
    by=['year_bs', 'month_bs', 'bfi_name']
)

# Convert value column to numeric float safely, coercing errors to NaN
nrb_balance_sheet['value_mn_npr'] = pd.to_numeric(nrb_balance_sheet['value_mn_npr'], errors='coerce')

# Drop rows where value couldn't be parsed as float
initial_count = len(nrb_balance_sheet)
# nrb_balance_sheet = nrb_balance_sheet.dropna(subset=['value_mn_npr'])
print(f"Retained {len(nrb_balance_sheet)} out of {initial_count} rows after numeric conversion.")

# Quick summary statistics to verify values
print(nrb_balance_sheet['value_mn_npr'].describe())
# 1. Ensure numeric conversion for values
nrb_balance_sheet['value_mn_npr'] = pd.to_numeric(nrb_balance_sheet['value_mn_npr'], errors='coerce')

# 2. Pivot the raw extracted data into a wide institutional panel structure
pivot_df = nrb_balance_sheet.pivot_table(
    index=['source_file', 'year_bs', 'month_bs', 'bfi_name'],
    columns='line_item',
    values='value_mn_npr',
    aggfunc='sum'
).reset_index()

# Flatten column multi-index if any
pivot_df.columns.name = None

# Helper to safely get columns
def get_col(df, col_name):
    # Match columns containing the target string case-insensitively
    matched = [c for c in df.columns if col_name.lower() in str(c).lower()]
    if matched:
        return df[matched[0]].fillna(0.0)
    return pd.Series(0.0, index=df.index)

# 3. Define top-level asset components for Size calculation (X_1)
asset_components = [
    'LIQUID FUNDS',
    'INVESTMENT IN SECURITIES',
    'SHARE & OTHER INVESTMENT',
    'LOANS & ADVANCES',
    'BILL PURCHASED',
    'FIXED ASSETS',
    'OTHER ASSETS'
]

size_series = pd.Series(0.0, index=pivot_df.index)
for comp in asset_components:
    matching_cols = [c for c in pivot_df.columns if comp.lower() in str(c).lower()]
    for mc in matching_cols:
        size_series += pivot_df[mc].fillna(0.0)

pivot_df['model_size'] = np.where(size_series > 0, size_series, np.nan)

# 4. Extract core variables
pivot_df['K_i'] = get_col(pivot_df, 'CAPITAL FUND')
pivot_df['liquid_funds'] = get_col(pivot_df, 'LIQUID FUNDS')
pivot_df['total_deposits'] = get_col(pivot_df, 'DEPOSITS')
pivot_df['loans_advances'] = get_col(pivot_df, 'LOANS & ADVANCES')
pivot_df['total_borrowings'] = get_col(pivot_df, 'BORROWINGS')
pivot_df['nrb_borrowing'] = get_col(pivot_df, 'a. NRB')

# Marginals (r_i, c_i)
pivot_df['r_i'] = get_col(pivot_df, 'Interbank Lending')
pivot_df['c_i'] = get_col(pivot_df, 'Interbank Borrowing')

# 5. Constructing the 5-Dimensional Covariate Vector X_{i,t}
pivot_df['X_1_size'] = pivot_df['model_size']
pivot_df['X_2_capital_ratio'] = pivot_df['K_i'] / pivot_df['model_size'].replace(0, np.nan)
pivot_df['X_3_liquidity_ratio'] = pivot_df['liquid_funds'] / pivot_df['model_size'].replace(0, np.nan)
pivot_df['X_4_ldr'] = pivot_df['loans_advances'] / pivot_df['total_deposits'].replace(0, np.nan)
pivot_df['X_5_nrb_dependence'] = pivot_df['nrb_borrowing'] / pivot_df['total_borrowings'].replace(0, np.nan)

# Clean ratio NaNs
ratio_cols = ['X_2_capital_ratio', 'X_3_liquidity_ratio', 'X_4_ldr', 'X_5_nrb_dependence']
for col in ratio_cols:
    pivot_df[col] = pivot_df[col].fillna(0.0)

# Final model-ready panel columns
model_panel_cols = [
    'source_file', 'year_bs', 'month_bs', 'bfi_name',
    'r_i', 'c_i', 'K_i',
    'X_1_size', 'X_2_capital_ratio', 'X_3_liquidity_ratio', 
    'X_4_ldr', 'X_5_nrb_dependence'
]

model_panel = pivot_df[[c for c in model_panel_cols if c in pivot_df.columns]].copy()
print(f"Successfully constructed model panel with {len(model_panel)} rows.")
model_panel.head(10)

In [1]:
#!/usr/bin/env -S uv run --script
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "torch",
# ]
# ///

print(f"CAPACITY PROJECTION TEST RESULTS")

import torch

try:
    from paper1_model.config import CapacityProjectionConfig
    from paper1_model.capacity_projection import capacitated_projection
except ImportError:
    from config import CapacityProjectionConfig
    from capacity_projection import capacitated_projection


def feasible_case() -> bool:
    torch.manual_seed(0)
    n = 8
    W_tilde = torch.rand(n, n) + 0.1
    r = W_tilde.sum(dim=1)
    c = W_tilde.sum(dim=0)
    cap = r.max() * 10 * torch.ones(n)  # generous, never binds

    W = capacitated_projection(W_tilde, r, c, cap, CapacityProjectionConfig())

    row_err = (W.sum(dim=1) - r).abs().max().item()
    col_err = (W.sum(dim=0) - c).abs().max().item()
    passed = row_err < 1e-3 and col_err < 1e-3
    print(f"[feasible_case] row_err={row_err:.2e} col_err={col_err:.2e} -> {'PASS' if passed else 'FAIL'}")
    return passed


def binding_cap_case() -> bool:
    """Unlike test_sinkhorn.py's old (retired) binding_cap_case, this checks a true
    PER-ENTRY cap: cap is set below several individual raw entries in rows 0/1, but
    comfortably above r_i/n, so hitting the row marginal by spreading mass across
    multiple entries is still feasible -- the thing the old row-scalar Sinkhorn clip
    could never actually do.
    """
    torch.manual_seed(1)
    n = 8
    W_tilde = torch.rand(n, n) + 0.1
    r = W_tilde.sum(dim=1)
    c = W_tilde.sum(dim=0)

    cap = r.max() * 10 * torch.ones(n)
    cap[0] = (r[0] / n) * 1.2  # feasible (cap*n > r), but below several raw entries
    cap[1] = (r[1] / n) * 1.5

    W = capacitated_projection(W_tilde, r, c, cap, CapacityProjectionConfig())

    max_entry_violation = (W - cap.unsqueeze(1)).max().item()
    cap_ok = max_entry_violation < 1e-4

    row_err = (W.sum(dim=1) - r).abs().max().item()
    col_err = (W.sum(dim=0) - c).abs().max().item()
    marginals_ok = row_err < 1e-3 and col_err < 1e-3

    n_pinned_row0 = (W[0] >= cap[0] - 1e-4).sum().item()
    n_pinned_row1 = (W[1] >= cap[1] - 1e-4).sum().item()

    passed = cap_ok and marginals_ok
    print(
        f"[binding_cap_case] max_entry_violation={max_entry_violation:.2e} "
        f"row_err={row_err:.2e} col_err={col_err:.2e} -> {'PASS' if passed else 'FAIL'}"
    )
    print(
        f"[binding_cap_case] entries pinned at cap: row0={n_pinned_row0}/{n} "
        f"row1={n_pinned_row1}/{n} (>1 confirms a true per-entry cap, not a row-sum "
        "cap -- cf. the retired Sinkhorn-clip finding)"
    )
    return passed


def infeasible_case() -> bool:
    """Informational: cap set so tight (cap_i * N < r_i) that no feasible W exists for
    row 0. Not expected to hit that row's marginal exactly -- only checks the cap
    itself is still respected (Dykstra converges to a least-violation compromise, not
    a silent cap breach).
    """
    torch.manual_seed(3)
    n = 8
    W_tilde = torch.rand(n, n) + 0.1
    r = W_tilde.sum(dim=1)
    c = W_tilde.sum(dim=0)

    cap = r.max() * 10 * torch.ones(n)
    cap[0] = (r[0] / n) * 0.3  # cap[0] * n < r[0]: provably infeasible for row 0

    W = capacitated_projection(W_tilde, r, c, cap, CapacityProjectionConfig())

    row0_shortfall = (W.sum(dim=1) - r).abs()[0].item()
    cap_violation = (W[0] - cap[0]).max().item()
    print(
        f"[infeasible_case] row0 shortfall={row0_shortfall:.4f} (cap*n="
        f"{cap[0].item() * n:.4f} < target r={r[0].item():.4f}, nonzero expected)"
    )
    cap_held = cap_violation < 1e-4
    print(
        f"[infeasible_case] max per-entry cap violation on row0={cap_violation:.2e} "
        f"-> {'PASS (cap held)' if cap_held else 'FAIL (cap breached)'}"
    )
    return cap_held


def gradient_flow_case() -> bool:
    torch.manual_seed(2)
    n = 6
    W_tilde = (torch.rand(n, n) + 0.1).requires_grad_(True)
    r = W_tilde.detach().sum(dim=1)
    c = W_tilde.detach().sum(dim=0)
    cap = r.max() * 10 * torch.ones(n)

    W = capacitated_projection(W_tilde, r, c, cap, CapacityProjectionConfig())
    W.sum().backward()

    grad = W_tilde.grad
    has_grad = grad is not None
    finite = has_grad and torch.isfinite(grad).all().item()
    passed = bool(has_grad and finite)
    print(f"[gradient_flow_case] grad_exists={has_grad} grad_finite={finite} -> {'PASS' if passed else 'FAIL'}")
    return passed


if __name__ == "__main__":
    results = {
        "feasible_case": feasible_case(),
        "binding_cap_case": binding_cap_case(),
        "infeasible_case": infeasible_case(),
        "gradient_flow_case": gradient_flow_case(),
    }
    print("\n=== SUMMARY ===")
    for name, passed in results.items():
        print(f"{name}: {'PASS' if passed else 'FAIL'}")
    print(f"\nOVERALL: {'PASS' if all(results.values()) else 'FAIL'}")


CAPACITY PROJECTION TEST RESULTS
[feasible_case] row_err=0.00e+00 col_err=0.00e+00 -> PASS
[binding_cap_case] max_entry_violation=0.00e+00 row_err=5.25e-06 col_err=1.43e-06 -> PASS
[binding_cap_case] entries pinned at cap: row0=4/8 row1=0/8 (>1 confirms a true per-entry cap, not a row-sum cap -- cf. the retired Sinkhorn-clip finding)
[infeasible_case] row0 shortfall=2.2889 (cap*n=0.9810 < target r=3.2699, nonzero expected)
[infeasible_case] max per-entry cap violation on row0=0.00e+00 -> PASS (cap held)
[gradient_flow_case] grad_exists=True grad_finite=True -> PASS

=== SUMMARY ===
feasible_case: PASS
binding_cap_case: PASS
infeasible_case: PASS
gradient_flow_case: PASS

OVERALL: PASS


In [ ]:
#!/usr/bin/env -S uv run --script
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "torch",
# ]
# ///

print("TEST SINKHORN RESULTS")
import torch

try:
    from paper1_model.config import SinkhornConfig
    from paper1_model.sinkhorn import sinkhorn_projection
except ImportError:
    from config import SinkhornConfig
    from sinkhorn import sinkhorn_projection


def feasible_case() -> bool:
    torch.manual_seed(0)
    n = 8
    W_tilde = torch.rand(n, n) + 0.1
    r = W_tilde.sum(dim=1)
    c = W_tilde.sum(dim=0)

    W = sinkhorn_projection(W_tilde, r, c, SinkhornConfig())

    row_err = (W.sum(dim=1) - r).abs().max().item()
    col_err = (W.sum(dim=0) - c).abs().max().item()
    passed = row_err < 1e-3 and col_err < 1e-3
    print(f"[feasible_case] row_err={row_err:.2e} col_err={col_err:.2e} -> {'PASS' if passed else 'FAIL'}")
    return passed


def gradient_flow_case() -> bool:
    torch.manual_seed(2)
    n = 6
    W_tilde = (torch.rand(n, n) + 0.1).requires_grad_(True)
    r = W_tilde.detach().sum(dim=1)
    c = W_tilde.detach().sum(dim=0)

    W = sinkhorn_projection(W_tilde, r, c, SinkhornConfig())
    W.sum().backward()

    grad = W_tilde.grad
    has_grad = grad is not None
    finite = has_grad and torch.isfinite(grad).all().item()
    passed = bool(has_grad and finite)
    print(f"[gradient_flow_case] grad_exists={has_grad} grad_finite={finite} -> {'PASS' if passed else 'FAIL'}")
    return passed


if __name__ == "__main__":
    results = {
        "feasible_case": feasible_case(),
        "gradient_flow_case": gradient_flow_case(),
    }
    print("\n=== SUMMARY (uncapped Sinkhorn/IPF -- Gandy-Veraart/RAS baseline only;")
    print("    the capped constraint layer is tested separately in")
    print("    test_capacity_projection.py) ===")
    for name, passed in results.items():
        print(f"{name}: {'PASS' if passed else 'FAIL'}")
    print(f"\nOVERALL: {'PASS' if all(results.values()) else 'FAIL'}")


TEST SINKHORN RESULTS
[feasible_case] row_err=4.77e-07 col_err=4.77e-07 -> PASS
[gradient_flow_case] grad_exists=True grad_finite=True -> PASS

=== SUMMARY (uncapped Sinkhorn/IPF -- Gandy-Veraart/RAS baseline only;
    the capped constraint layer is tested separately in
    test_capacity_projection.py) ===
feasible_case: PASS
gradient_flow_case: PASS

OVERALL: PASS
